In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import coremltools as ct
from coremltools.converters.mil import Builder as mb

pd.set_option("display.max_rows", 3000)
pd.options.display.float_format = "{:,.3g}".format

## Prepare test model

In [ ]:
# pyright: reportGeneralTypeIssues=false, reportAttributeAccessIssue=false
input_channels = 32
input_size = 256


@mb.program(
    input_specs=[
        mb.TensorSpec(
            shape=(1, input_channels, input_size, input_size),
            dtype=ct.converters.mil.mil.types.fp16,
        ),
        mb.TensorSpec(
            shape=(1, input_channels, input_size, input_size),
            dtype=ct.converters.mil.mil.types.fp16,
        ),
    ],
    opset_version=ct.target.macOS14,
)
def prog(conv_input, act_input):
    # CoreML does not run very small models on NPU, add convolution and relu to make it run on NPU
    conv_out = mb.conv(
        x=conv_input,
        weight=np.random.rand(1, input_channels, 3, 3),
        bias=np.random.rand(1),
        name="conv",
    )
    conv_out = mb.relu(x=conv_out, name="conv_out")

    relu_out = mb.relu(x=act_input, name="relu")
    silu_out = mb.silu(x=act_input, name="silu")
    sigmoid_out = mb.sigmoid(x=act_input, name="sigmoid")
    x_mul_sigmoid_out = mb.mul(x=act_input, y=sigmoid_out, name="x_mul_sigmoid")
    leaky_relu_out = mb.leaky_relu(x=act_input, alpha=0.01, name="leaky_relu")
    tanh_out = mb.tanh(x=act_input, name="tanh")
    gelu_sigmoid_approx_out = mb.gelu(x=act_input, mode="SIGMOID_APPROXIMATION", name="gelu_sigmoid_approx")

    return (
        conv_out,
        relu_out,
        silu_out,
        sigmoid_out,
        x_mul_sigmoid_out,
        leaky_relu_out,
        tanh_out,
        gelu_sigmoid_approx_out,
    )


print(prog)
model = ct.convert(prog, minimum_deployment_target=ct.target.macOS14)
test_model_path = "./output/act_test_model.mlpackage"
model.save(test_model_path)  # type: ignore

## Compute activation function values

In [ ]:
conv_input = np.random.rand(1, input_channels, input_size, input_size).astype(np.float16)
act_input = np.linspace(-5.0, 5.0, input_channels * input_size * input_size).reshape(
    1, input_channels, input_size, input_size
)
model_input = {"conv_input": conv_input, "act_input": act_input}

cpu_model = ct.models.MLModel(test_model_path, compute_units=ct.ComputeUnit.CPU_ONLY)
npu_model = ct.models.MLModel(test_model_path, compute_units=ct.ComputeUnit.CPU_AND_NE)

results = {}
results["Torch (CPU)"] = {
    "relu": torch.nn.functional.relu(torch.tensor(act_input)).numpy().astype(np.float16),
    "leaky_relu": torch.nn.functional.leaky_relu(torch.tensor(act_input), negative_slope=0.01)
    .numpy()
    .astype(np.float16),
    "silu": torch.nn.functional.silu(torch.tensor(act_input)).numpy().astype(np.float16),
    "sigmoid": torch.nn.functional.sigmoid(torch.tensor(act_input)).numpy().astype(np.float16),
    "x_mul_sigmoid": (torch.tensor(act_input) * torch.nn.functional.sigmoid(torch.tensor(act_input)))
    .numpy()
    .astype(np.float16),
    "tanh": torch.nn.functional.tanh(torch.tensor(act_input)).numpy().astype(np.float16),
    "gelu_sigmoid_approx": (torch.tensor(act_input) * torch.nn.functional.sigmoid(1.702 * torch.tensor(act_input)))
    .numpy()
    .astype(np.float16),
}
results["CoreML (CPU)"] = cpu_model.predict(model_input)
results["CoreML (NPU)"] = npu_model.predict(model_input)

## Visualize activation errors

In [ ]:
x = act_input.flatten()

target_device = "Torch (CPU)"
function_names = list(results[target_device].keys())

fig, axs = plt.subplots(len(function_names), 2, figsize=(15, 15), sharex=True)
for i, func_name in enumerate(function_names):
    desired_output = results[target_device][f"{func_name}"].flatten()
    ax1 = axs[i, 0]
    ax2 = axs[i, 1]

    for j, (device, res) in enumerate(results.items()):
        if f"{func_name}" not in res:
            continue
        out = res[f"{func_name}"].flatten()
        diff = out - desired_output
        ax1.plot(x, out, label=f"{device}", lw=1, color=f"C{j}")
        if device != target_device:
            ax2.plot(x, diff, label=f"{device}", lw=1, color=f"C{j}")

    ax1.set_ylabel(f"{func_name}")
    ax2.set_ylabel(f"{func_name} error")
    ax2.set_ylim(-0.02, 0.02)
    ax1.legend(loc=2)
    ax2.legend(loc=2)

fig.subplots_adjust(hspace=0.1)

## SiLU activation

In [ ]:
plt.figure(figsize=(15, 8))
act_function = "silu"
for device, res in results.items():
    out = res.get(act_function)
    if out is None:
        continue
    plt.plot(x, out.flatten(), label=f"{device}")
plt.legend()
plt.ylabel(f"{act_function}")
plt.xlabel("Input")
plt.grid(True)
plt.xlim(-1, 1)
plt.ylim(-0.5, 1)